# 02 · Source adaptation, splitting and training-only preprocessing

The adapter renames amt to Amount and converts transaction timestamps to elapsed seconds. It keeps category, state, population and coordinates. Raw names, card numbers, identifiers, dates of birth and unix_time are not model inputs. The same feature engineering is used at inference.

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from src.dataset import load_training_frame
from src.data_processing import inspect_dataset, split_dataset, make_schema, validate_features
metadata = json.loads(Path('models/model_metadata.json').read_text())
frame, target, source = load_training_frame(Path('data/raw') / metadata['dataset']['file'])
train, validation, test = split_dataset(frame, target, metadata['training']['seed'], metadata['dataset'].get('split_strategy', 'stratified'))
features = [f['name'] for f in metadata['features']]

schema = metadata['features']
assert set(train.index).isdisjoint(validation.index)
assert set(train.index).isdisjoint(test.index)
assert set(validation.index).isdisjoint(test.index)
if metadata['dataset']['split_strategy'] == 'chronological':
    assert train.Time.max() < validation.Time.min()
    assert validation.Time.max() < test.Time.min()
display(pd.DataFrame([{'partition': name, 'rows': len(part), 'fraud': int(part[target].sum())} for name, part in [('train', train), ('validation', validation), ('test', test)]]))
display(pd.DataFrame(schema))


,partition,rows,fraud
0,train,629145,3712
1,validation,209715,1149
2,test,209715,1145


,name,type,default,minimum_observed,maximum_observed,nullable,categories
0,Amount,number,47.4,1.000000,2.654412e+04,True,NaN
1,Time,number,13067340.0,0.000000,2.300490e+07,True,NaN
2,category,category,gas_transport,NaN,NaN,True,"[entertainment, food_dining, gas_transport, gr..."
3,state,category,TX,NaN,NaN,True,"[AK, AL, AR, AZ, CA, CO, CT, DC, DE, FL, GA, H..."
4,city_pop,number,2456.0,23.000000,2.906700e+06,True,NaN
5,lat,number,39.3543,20.027100,6.669330e+01,True,NaN
6,long,number,-87.4616,-165.672300,-6.795030e+01,True,NaN
7,merch_lat,number,39.366152,19.029798,6.751027e+01,True,NaN
8,merch_long,number,-87.415668,-166.671242,-6.695654e+01,True,NaN


## Shared transformations
Median imputation, robust scaling and float32 arrays are fitted on training rows. Categorical values use unknown-safe one-hot encoding. The geographic schema adds great-circle merchant distance; elapsed time adds daily and weekly cyclic features. Log amount is derived inside the pipeline.

In [2]:
from src.feature_engineering import preprocessing
pipeline = preprocessing(schema).fit(validate_features(train[features], schema))
transformed = pipeline.transform(validation[features].head(5))
print('Input:', len(features), 'Transformed:', transformed.shape[1], 'dtype:', transformed.dtype)
print(pipeline.get_feature_names_out())
missing = validation[features].head(1).copy()
missing.loc[:, 'Amount'] = np.nan
assert np.isfinite(pipeline.transform(missing)).all()


Input: 9 Transformed: 78 dtype: float32
['Amount' 'Time' 'city_pop' 'lat' 'long' 'merch_lat' 'merch_long'
 'Amount_log1p' 'Time_phase_sin' 'Time_phase_cos' 'merchant_distance_km'
 'Time_week_sin' 'Time_week_cos' 'category_entertainment'
 'category_food_dining' 'category_gas_transport' 'category_grocery_net'
 'category_grocery_pos' 'category_health_fitness' 'category_home'
 'category_kids_pets' 'category_misc_net' 'category_misc_pos'
 'category_personal_care' 'category_shopping_net' 'category_shopping_pos'
 'category_travel' 'state_AK' 'state_AL' 'state_AR' 'state_AZ' 'state_CA'
 'state_CO' 'state_CT' 'state_DC' 'state_DE' 'state_FL' 'state_GA'
 'state_HI' 'state_IA' 'state_ID' 'state_IL' 'state_IN' 'state_KS'
 'state_KY' 'state_LA' 'state_MA' 'state_MD' 'state_ME' 'state_MI'
 'state_MN' 'state_MO' 'state_MS' 'state_MT' 'state_NC' 'state_ND'
 'state_NE' 'state_NH' 'state_NJ' 'state_NM' 'state_NV' 'state_NY'
 'state_OH' 'state_OK' 'state_OR' 'state_PA' 'state_RI' 'state_SC'
 'state_SD' '

## Oversampling
Random oversampling duplicates complete minority-class training rows and preserves categorical combinations. It runs inside each training CV fold, after training-fitted preprocessing. Validation/test rows retain their natural prevalence. Oversampled classifiers do not also apply class weights.

In [3]:
display(pd.DataFrame([{'model': row['name'], 'strategy': row['imbalance_strategy'], 'parameters': row['parameters']} for row in metadata['comparison']]))
display(pd.DataFrame(metadata['training_class_counts']).rename(index={'0': 'Legitimate', '1': 'Fraud'}))
print(metadata['training']['cv_strategy'])


,model,strategy,parameters
0,XGBoost,unweighted boosting baseline,{'classifier__max_depth': 3}
1,XGBoost · random oversampling,training-only random oversampling; valid categ...,"{'classifier__max_depth': 5, 'sampler__samplin..."


,before,after
Legitimate,625433,625433
Fraud,3712,156358


expanding chronological windows


## Saved-model parity
Sampling is skipped at prediction time. The standalone preprocessor must produce the same probabilities as the complete saved pipeline.

In [4]:
import joblib
from backend.services.model_service import ModelService
model = ModelService(Path('models'))
saved_preprocessor = joblib.load('models/preprocessor.joblib')
probe = test[features].head(20)
direct = model.pipeline.named_steps['classifier'].predict_proba(saved_preprocessor.transform(probe))[:, 1]
np.testing.assert_allclose(direct, model.predict_proba(probe), rtol=1e-7)
print('Saved preprocessor and inference pipeline agree on 20 later test examples')


Saved preprocessor and inference pipeline agree on 20 later test examples
